# EntreDegraus — Dynamic Programming | Challenge Motiva

**Dados reais de ocorrências de manutenção (2025–2026), e NÃO dados de sensores.** Os arquivos Motiva fornecidos incluem registros de falhas/avarias, com identificação do equipamento, data, duração e classificações. **Não há colunas estruturadas de vibração, temperatura, corrente, tensão, óleo ou velocidade nas planilhas `ecr_2025.XLSX` e `ecr_2026.XLSX`.**

Por isso, a pontuação é uma **proxy didática de criticidade por ocorrências**, e não degradação física estimada por sensores. A rede de deslocamentos e os tempos também são **hipóteses**, identificadas no JSON. Não usar o plano para operação real.

In [1]:
from pathlib import Path
import pandas as pd
from entredegraus_dp import (carregar_dados, avaliar_algoritmos, construir_grafo, comparar_rotas, planejar_turno, BASE, degradacao_recursiva, degradacao_memoizada, degradacao_tabulada)
leituras, rede = carregar_dados('dados')
print('Ocorrências:', len(leituras), '| Escadas:', leituras.escada_id.nunique(), '| Estações:', leituras.estacao.nunique())
display(leituras[['escada_id','estacao','momento','duracao_indisponibilidade_min','indisponivel','falha']].head())

Ocorrências: 17951 | Escadas: 197 | Estações: 12


,escada_id,estacao,momento,duracao_indisponibilidade_min,indisponivel,falha
0,SP04-6-ECR-BUT-ER01,BUT,2025-01-03 13:56:42,5.30,0,0
1,SP04-6-ECR-BUT-ER01,BUT,2025-01-08 14:46:31,12.92,1,0
2,SP04-6-ECR-BUT-ER01,BUT,2025-01-10 06:28:33,14.37,1,0
3,SP04-6-ECR-BUT-ER01,BUT,2025-01-11 10:32:33,6.40,0,0
4,SP04-6-ECR-BUT-ER01,BUT,2025-01-14 05:06:48,8.53,1,0


## 1. Programação dinâmica — 3 versões

Recorrência: **D(i) = dano(i) + 0,85 D(i−1) + 0,15 D(i−2)**, com índices negativos valendo zero. O dano é calculado por ocorrência real com pesos ilustrativos (veja a função `dano_leitura`). Para tornar viável a recursão pura, todas as três versões são medidas sobre **as mesmas últimas 16 ocorrências por equipamento**. Recursão pura é exponencial; memoização e tabulação são lineares. O ranking usa esse recorte, não todo o histórico.

In [2]:
ranking, tempos = avaliar_algoritmos(leituras, repeticoes=2, janela=16)
display(ranking.head(15))
display(tempos.groupby('algoritmo',as_index=False).tempo_ms.agg(['mean','median','max']))
print('Interpretação: as três funções retornam o mesmo valor; a recursão pura repete subproblemas, enquanto memoização e tabulação os evitam.')

,posicao,escada_id,estacao,criticidade,ocorrencias_na_janela,ocorrencias_totais
0,1,SP04-6-ECR-PIN-ER06,PIN,57.524114,16,135
1,2,SP04-6-ECR-PIN-ER21,PIN,55.587541,16,166
2,3,SP04-6-ECR-PIN-ER14,PIN,53.632263,16,143
3,4,SP04-6-ECR-PIN-ER26,PIN,50.981173,16,68
4,5,SP04-6-ECR-PIN-ER20,PIN,50.697164,16,63
5,6,SP04-6-ECR-PIN-ER30,PIN,49.061287,16,68
6,7,SP04-6-ECR-PIN-ER22,PIN,49.019341,16,174
7,8,SP04-6-ECR-PIN-ER09,PIN,47.843083,16,103
8,9,SP04-6-ECR-PIN-ER24,PIN,46.637110,16,371
9,10,SP04-6-ECR-MBI-ER12,MBI,46.014881,16,86


,algoritmo,mean,median,max
0,Iterativa (tabulação),0.000956,0.000951,0.001342
1,Memoizada (dicionário),0.003969,0.003725,0.020560
2,Recursiva pura,0.257143,0.241838,0.590873


Interpretação: as três funções retornam o mesmo valor; a recursão pura repete subproblemas, enquanto memoização e tabulação os evitam.


## 2. Grafo, BFS e Dijkstra

Os **12 códigos de estação** são extraídos do campo `Local de Instalação` dos dados reais. A **base `BASE_SIMULADA`, as arestas e os minutos** foram construídos apenas para o exercício e **não são itinerários ou tempos reais da Motiva**. BFS minimiza trechos; Dijkstra minimiza minutos.

In [3]:
grafo=construir_grafo(rede)
rotas=comparar_rotas(grafo,rede,BASE)
display(rotas)
print('Interpretação: menos trechos não significa necessariamente menor tempo. As rotas são válidas apenas para o grafo didático.')

,estacao,trechos_bfs,minutos_bfs,minutos_dijkstra,minutos_a_estrela,rota_bfs,rota_dijkstra,rota_a_estrela
0,LUZ,1,7.0,7.0,7.0,BASE_SIMULADA → LUZ,BASE_SIMULADA → LUZ,BASE_SIMULADA → LUZ
1,REP,2,11.0,11.0,11.0,BASE_SIMULADA → LUZ → REP,BASE_SIMULADA → LUZ → REP,BASE_SIMULADA → LUZ → REP
2,PIN,2,15.0,15.0,15.0,BASE_SIMULADA → LUZ → PIN,BASE_SIMULADA → LUZ → PIN,BASE_SIMULADA → LUZ → PIN
3,FRA,3,21.0,21.0,21.0,BASE_SIMULADA → LUZ → PIN → FRA,BASE_SIMULADA → LUZ → PIN → FRA,BASE_SIMULADA → LUZ → PIN → FRA
4,FAL,3,24.0,24.0,24.0,BASE_SIMULADA → LUZ → PIN → FAL,BASE_SIMULADA → LUZ → PIN → FAL,BASE_SIMULADA → LUZ → PIN → FAL
5,BUT,4,28.0,28.0,28.0,BASE_SIMULADA → LUZ → PIN → FAL → BUT,BASE_SIMULADA → LUZ → PIN → FAL → BUT,BASE_SIMULADA → LUZ → PIN → FAL → BUT
6,PTA,4,34.0,33.0,33.0,BASE_SIMULADA → LUZ → PIN → FAL → PTA,BASE_SIMULADA → LUZ → PIN → FAL → BUT → PTA,BASE_SIMULADA → LUZ → PIN → FAL → BUT → PTA
7,HIG,5,34.0,34.0,34.0,BASE_SIMULADA → LUZ → PIN → FAL → BUT → HIG,BASE_SIMULADA → LUZ → PIN → FAL → BUT → HIG,BASE_SIMULADA → LUZ → PIN → FAL → BUT → HIG
8,MBI,5,43.0,41.0,41.0,BASE_SIMULADA → LUZ → PIN → FAL → PTA → MBI,BASE_SIMULADA → LUZ → PIN → FAL → BUT → HIG → MBI,BASE_SIMULADA → LUZ → PIN → FAL → BUT → HIG → MBI
9,VSO,6,47.0,45.0,45.0,BASE_SIMULADA → LUZ → PIN → FAL → PTA → MBI → VSO,BASE_SIMULADA → LUZ → PIN → FAL → BUT → HIG → ...,BASE_SIMULADA → LUZ → PIN → FAL → BUT → HIG → ...


Interpretação: menos trechos não significa necessariamente menor tempo. As rotas são válidas apenas para o grafo didático.


## 3. A* comparado ao Dijkstra

A heurística é `h(v) = distância_euclidiana(v, destino) × min(tempo_da_aresta/distância_da_aresta)`. As coordenadas são **ilustrativas**. Pelo limite inferior calculado a partir de todas as arestas e pela desigualdade triangular, a heurística é admissível e consistente no grafo fornecido. Assim, A* e Dijkstra devem encontrar **o mesmo custo ótimo**; diferenças de eficiência dependem do grafo. O tempo de execução pode ser muito pequeno e sujeito a ruído.

In [4]:
assert (rotas.minutos_dijkstra-rotas.minutos_a_estrela).abs().max()<1e-9
print('Dijkstra e A*: custo igual em',len(rotas),'destinos.')
display(rotas[['estacao','minutos_dijkstra','minutos_a_estrela','rota_a_estrela']])
print('Interpretação: a heurística não altera o custo mínimo; o exemplo não demonstra superioridade universal do A*.')

Dijkstra e A*: custo igual em 12 destinos.


,estacao,minutos_dijkstra,minutos_a_estrela,rota_a_estrela
0,LUZ,7.0,7.0,BASE_SIMULADA → LUZ
1,REP,11.0,11.0,BASE_SIMULADA → LUZ → REP
2,PIN,15.0,15.0,BASE_SIMULADA → LUZ → PIN
3,FRA,21.0,21.0,BASE_SIMULADA → LUZ → PIN → FRA
4,FAL,24.0,24.0,BASE_SIMULADA → LUZ → PIN → FAL
5,BUT,28.0,28.0,BASE_SIMULADA → LUZ → PIN → FAL → BUT
6,PTA,33.0,33.0,BASE_SIMULADA → LUZ → PIN → FAL → BUT → PTA
7,HIG,34.0,34.0,BASE_SIMULADA → LUZ → PIN → FAL → BUT → HIG
8,MBI,41.0,41.0,BASE_SIMULADA → LUZ → PIN → FAL → BUT → HIG → MBI
9,VSO,45.0,45.0,BASE_SIMULADA → LUZ → PIN → FAL → BUT → HIG → ...


Interpretação: a heurística não altera o custo mínimo; o exemplo não demonstra superioridade universal do A*.


## 4. Plano de manutenção integrado

`planejar_turno(base, horas)` prioriza as escadas por pontuação e calcula deslocamentos mínimos entre atendimentos, incluindo retorno à base. **45 minutos por serviço** são hipótese didática. A seleção é gulosa, **não garante solução global ótima** e não representa uma ordem de serviço real.

In [5]:
plano, resumo=planejar_turno(BASE,8,ranking,grafo,rede)
display(plano)
print('Resumo:',resumo)
assert resumo['minutos_livres']>=-1e-8
print('Interpretação: o plano cabe em oito horas contando o retorno à base; sua prioridade depende de uma proxy histórica, não de medições de sensores.')

,ordem,escada_id,estacao,criticidade,origem,rota,deslocamento_min,servico_min,acumulado_min
0,1,SP04-6-ECR-PIN-ER06,PIN,57.5241,BASE_SIMULADA,BASE_SIMULADA → LUZ → PIN,15.0,45,60.0
1,2,SP04-6-ECR-PIN-ER21,PIN,55.5875,PIN,PIN,0.0,45,105.0
2,3,SP04-6-ECR-PIN-ER14,PIN,53.6323,PIN,PIN,0.0,45,150.0
3,4,SP04-6-ECR-PIN-ER26,PIN,50.9812,PIN,PIN,0.0,45,195.0
4,5,SP04-6-ECR-PIN-ER20,PIN,50.6972,PIN,PIN,0.0,45,240.0
5,6,SP04-6-ECR-PIN-ER30,PIN,49.0613,PIN,PIN,0.0,45,285.0
6,7,SP04-6-ECR-PIN-ER22,PIN,49.0193,PIN,PIN,0.0,45,330.0
7,8,SP04-6-ECR-PIN-ER09,PIN,47.8431,PIN,PIN,0.0,45,375.0
8,9,SP04-6-ECR-PIN-ER24,PIN,46.6371,PIN,PIN,0.0,45,420.0
9,10,SP04-6-ECR-PIN-ER11,PIN,45.9440,PIN,PIN,0.0,45,465.0


Resumo: {'base': 'BASE_SIMULADA', 'horas_disponiveis': 8, 'minutos_usados_com_retorno': 480.0, 'minutos_livres': 0.0, 'rota_retorno': 'PIN → LUZ → BASE_SIMULADA', 'observacao': 'Heurística gulosa; prioriza criticidade, não garante ótimo global.'}
Interpretação: o plano cabe em oito horas contando o retorno à base; sua prioridade depende de uma proxy histórica, não de medições de sensores.


## 5. Limitações e rastreabilidade

Fonte: arquivos enviados `Dados Motiva/ecr_2025.XLSX` e `Dados Motiva/ecr_2026.XLSX`. O CSV preserva as colunas selecionadas e indica o ano de origem. Eventos com código de equipamento ou datas inválidas foram descartados na transformação. A rede, a base, os tempos, a função de pontuação e a duração dos serviços são **suposições acadêmicas**. Para cumprir literalmente o requisito de **degradação por sensores**, é necessário receber uma base com as seis variáveis físicas e substituir a função de dano; os algoritmos DP continuam reutilizáveis.